# Lesson 25 — Robust Optimization

## Learning objectives

1. State the robust-counterpart paradigm.
2. Recognize box, ellipsoidal, and budget uncertainty sets.
3. Reformulate robust LPs as deterministic problems.
4. Distinguish static from adjustable RO.

## 1. Robust counterparts

For an LP $\min c^\top x$ s.t. $a_i^\top x \le b_i$ where $a_i \in \mathcal{U}_i$ (uncertainty set centred on the *nominal* $\hat a_i$), the **robust counterpart** is

$$\min c^\top x \;\;\text{s.t.}\;\; a_i^\top x \le b_i \;\; \forall a_i \in \mathcal{U}_i.$$

Equivalent: $\sup_{a \in \mathcal U_i} a^\top x \le b_i$, a *semi-infinite* constraint. Tractability hinges on reformulating this supremum {cite:p}`BenTal2009,BertsimasSim2004`.

> **Notation note.** This lesson uses $\hat a$ for the *nominal* (centre) value of an uncertain coefficient — distinct from $\overline a / \underline a$ which the rest of the course reserves for upper / lower *bounds*.

## 2. Common uncertainty sets

- **Box** $\{a : |a_i - \hat a_i| \le \delta_i\}$ → reformulates to LP. Conservative.
- **Ellipsoid** $\{\hat a + Pu : \|u\| \le 1\}$ → SOCP {cite:p}`BenTal1999`.
- **Budget** {cite:p}`BertsimasSim2004` $\{a : \sum_i |a_i - \hat a_i|/\delta_i \le \Gamma\}$ → LP. Adjustable conservatism via $\Gamma$.

## 3. Robust reformulations

**Box uncertainty:**

$$\sup_{|\Delta_i| \le \delta_i} (\hat a + \Delta)^\top x = \hat a^\top x + \sum_i \delta_i |x_i| \le b.$$

Linearize $|x_i|$ with $x = x^+ - x^-, x^\pm \ge 0$.

**Ellipsoidal:**

$$\hat a^\top x + \|P^\top x\|_2 \le b.$$

SOCP.

**Budget {cite:p}`BertsimasSim2004`:**

$$\hat a^\top x + \max_{S \subseteq [n], |S| \le \Gamma} \sum_{i \in S} \delta_i |x_i| \le b.$$

Reformulates to an LP with $O(n)$ extra variables.

## 4. Adjustable RO

**Static RO:** $x$ chosen before $\xi$ realized.
**Adjustable RO:** $y(\xi)$ chosen after $\xi$, can depend on it. Affine decision rules $y(\xi) = y_0 + Y\xi$ keep tractability {cite:p}`BenTalGoryashko2004`.

Multi-stage RO is harder but surfaces in inventory, energy, and scheduling problems.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do

# Robust LP: min c'x s.t. a'x >= b for all a in box around the nominal hat_a.
n = 5
np.random.seed(0)
hat_a = np.random.uniform(1, 5, size=n)            # nominal coefficients
delta = 0.2 * hat_a                                 # 20 % uncertainty
b = 30.0
c = np.random.uniform(1, 3, size=n)

m = do.Model("robust_lp")
x = m.add_variables(n, lb=0, ub=10, name="x")
# Robust counterpart for >= b is hat_a' x - sum delta_i * |x_i| >= b.
xp = m.add_variables(n, lb=0); xm = m.add_variables(n, lb=0)
m.add_constraints(xp - xm == x)
m.add_constraint(hat_a @ x - delta @ (xp + xm) >= b)
m.minimize(c @ x)
r = m.solve()
print("robust opt:", r.objective, "  x:", r.x.round(2))


## 5. RO vs SP

|  | RO | SP |
|--|----|----|
| Uncertainty | Worst case in $\mathcal U$ | Probability distribution |
| Solution | Min over $x$, max over $a$ | Min expected cost |
| Tractability | Often LP/SOCP | LP with many scenarios |
| Conservatism | Can be high; tunable via $\Gamma$ | Tunable via risk measure |

A mature robust optimizer reaches for both — RO when uncertainty is bounded but distributions are unknown, SP when distributions are reliable.

## References
{cite:p}`BenTal2009,BertsimasSim2004,BenTal1999,BenTalGoryashko2004,Bertsimas2010`.